# Blinded pharmacogenomics study — `corpus_only/rep3`

Maps 110 blinded cases (gene + drug + genotype description) to **diplotype → CPIC
metaboliser/function phenotype → guideline recommendation**.

Inputs: `cases_blinded.json`, `cpic_guideline_corpus.json` (CPIC snapshot 2026-05-09).
No network access was used. All mapping logic lives in `pgx_pipeline.py`, which this
notebook imports and drives, so the notebook and the script cannot drift apart.

In [1]:
import json, re, os, sys, collections
import pandas as pd

HERE = os.getcwd()
cases  = json.load(open(os.path.join(HERE, "cases_blinded.json")))
corpus = json.load(open(os.path.join(HERE, "cpic_guideline_corpus.json")))
print("cases:", len(cases))
print("corpus schema:", corpus["schema_version"], "| generated:", corpus["generated_at_utc"])
print("genes:", len(corpus["genes"]))

cases: 110
corpus schema: v3.1 | generated: 2026-05-10T04:40:15Z
genes: 21


## 1. Corpus inventory

What the supplied corpus does and does not contain. This determines which mapping steps
can be grounded in the corpus and which must come from the analyst's own knowledge.

In [2]:
inv = pd.DataFrame([
    {"gene": g, "chars": v["char_count"], "drugs_covered": ", ".join(v["drugs_covered"]),
     "n_source_annotations": len(v["sources"])}
    for g, v in corpus["genes"].items()])
display(inv.sort_values("gene").reset_index(drop=True))

ALL_TEXT = "\n".join(v["guideline_excerpt"] for v in corpus["genes"].values())
case_rsids = sorted(set(re.findall(r"rs\d+", json.dumps(cases))))
present = [r for r in case_rsids if r in ALL_TEXT]
print("rsIDs used by the cases :", len(case_rsids))
print("of those, found in corpus:", present)
print("allele-definition tables in corpus:", "CYP2C9 only (*1/*2/*3 via rs1799853/rs1057910)")

rsIDs used by the cases : 24
of those, found in corpus: ['rs1057910', 'rs113993960', 'rs12979860', 'rs4149056', 'rs75527207', 'rs9923231']
allele-definition tables in corpus: CYP2C9 only (*1/*2/*3 via rs1799853/rs1057910)


,gene,chars,drugs_covered,n_source_annotations
0,CFTR,2799,ivacaftor,1
1,CYP2B6,2252,efavirenz,1
2,CYP2C19,25574,"citalopram, clopidogrel, omeprazole, voriconazole",5
3,CYP2C9,11593,"celecoxib, phenytoin, warfarin",3
4,CYP2D6,24781,"amitriptyline, codeine, ondansetron, paroxetine, tamoxifen",5
5,CYP3A5,1896,tacrolimus,1
6,CYP4F2,935,warfarin,1
7,DPYD,6061,"capecitabine, fluorouracil",2
8,G6PD,8699,"primaquine, rasburicase",2
9,HLA-A*31:01,3847,carbamazepine,1


## 2. Run the mapping pipeline

`pgx_pipeline.py` holds three explicit rule tables (assay→allele, diplotype→phenotype,
phenotype→recommendation) and asserts that **every recommendation string is a verbatim
substring of the supplied corpus**. Import executes it and writes all output files.

In [3]:
sys.path.insert(0, HERE)
import pgx_pipeline as P

cases=110  abstained=2  lethal_class=22
corpus sha256 (cases): bcda9e79e870a15e
corpus generated_at_utc: 2026-05-10T04:40:15Z | schema: v3.1


## 3. Table 1 — assay result → allele call

In [4]:
t1 = pd.read_csv("table_variant_to_allele.csv")
print(t1.basis.value_counts().to_dict())
display(t1[["gene","assay_result_pattern","allele_call","basis"]])

{'analyst-knowledge': 37, 'corpus': 14, 'case-annotation': 10}


,gene,assay_result_pattern,allele_call,basis
0,CYP2D6,rs3892097 T/T,*4/*4,analyst-knowledge
1,CYP2D6,rs3892097 C/T + rs1065852 G/A,*1/*4,analyst-knowledge
2,CYP2D6,rs3892097 C/T,*1/*4,analyst-knowledge
3,CYP2D6,All CYP2D6 SNPs homozygous reference,*1/*1,analyst-knowledge
4,CYP2D6,*1xN duplication,*1/*1xN,case-annotation
5,CYP2C19,rs4244285 A/A,*2/*2,analyst-knowledge
6,CYP2C19,rs4244285 G/A,*1/*2,analyst-knowledge
7,CYP2C19,rs12248560 T/T,*17/*17,analyst-knowledge
8,CYP2C19,rs12248560 C/T,*1/*17,analyst-knowledge
9,CYP2C19,All CYP2C19 SNPs homozygous reference,*1/*1,analyst-knowledge


## 4. Table 2 — diplotype → phenotype

In [5]:
t2 = pd.read_csv("table_diplotype_to_phenotype.csv")
display(t2)

,gene,diplotype,activity_score,phenotype
0,CYP2D6,*4/*4,0,CYP2D6 poor metabolizer
1,CYP2D6,*1/*4,1.0,CYP2D6 intermediate metabolizer
2,CYP2D6,*1/*1,2.0,CYP2D6 normal metabolizer
3,CYP2D6,*1/*1xN,>2.25,CYP2D6 ultrarapid metabolizer
4,CYP2C19,*2/*2,NaN,CYP2C19 poor metabolizer
5,CYP2C19,*1/*2,NaN,CYP2C19 intermediate metabolizer
6,CYP2C19,*1/*1,NaN,CYP2C19 normal metabolizer
7,CYP2C19,*1/*17,NaN,CYP2C19 rapid metabolizer
8,CYP2C19,*17/*17,NaN,CYP2C19 ultrarapid metabolizer
9,CYP2C9,*3/*3,0,CYP2C9 poor metabolizer


## 5. Table 3 — (gene, drug, phenotype) → recommendation

In [6]:
t3 = pd.read_csv("table_phenotype_to_recommendation.csv")
print("gene-drug-phenotype rules:", len(t3))
display(t3[["gene","drug","phenotype","action","guideline_source","abstained"]])

gene-drug-phenotype rules: 109


,gene,drug,phenotype,action,guideline_source,abstained
0,CYP2D6,codeine,CYP2D6 ultrarapid metabolizer,codeine: avoid - risk of serious morphine toxicity; if an opioid is warranted use a non-tramadol opioid,CPIC PA166104996,False
1,CYP2D6,codeine,CYP2D6 normal metabolizer,codeine: use label-recommended age- or weight-specific dosing,CPIC PA166104996,False
2,CYP2D6,codeine,CYP2D6 intermediate metabolizer,"codeine: use label-recommended age- or weight-specific dosing; if no response and an opioid is warranted, consider a non-tramadol opioid",CPIC PA166104996,False
3,CYP2D6,codeine,CYP2D6 poor metabolizer,codeine: avoid - diminished analgesia; if an opioid is warranted use a non-tramadol opioid,CPIC PA166104996,False
4,CYP2D6,tamoxifen,CYP2D6 ultrarapid metabolizer,tamoxifen: standard dosing 20 mg/day; avoid moderate/strong CYP2D6 inhibitors,CPIC PA166176068,False
5,CYP2D6,tamoxifen,CYP2D6 normal metabolizer,tamoxifen: standard dosing 20 mg/day; avoid moderate/strong CYP2D6 inhibitors,CPIC PA166176068,False
6,CYP2D6,tamoxifen,CYP2D6 intermediate metabolizer,tamoxifen: consider an aromatase inhibitor instead (plus ovarian suppression if premenopausal); if an AI is contraindicated consider tamoxifen 40 mg/day and avoid CYP2D6 inhibitors,CPIC PA166176068,False
7,CYP2D6,tamoxifen,CYP2D6 poor metabolizer,"tamoxifen: avoid - use alternative hormonal therapy (aromatase inhibitor, plus ovarian suppression if premenopausal)",CPIC PA166176068,False
8,CYP2D6,ondansetron,CYP2D6 ultrarapid metabolizer,ondansetron: avoid - select an alternative antiemetic not predominantly metabolised by CYP2D6 (e.g. granisetron),CPIC PA166161954,False
9,CYP2D6,ondansetron,CYP2D6 normal metabolizer,ondansetron: initiate therapy with the recommended starting dose,CPIC PA166161954,False


## 6. Answers

In [7]:
ans = pd.read_csv("claude_science_answers.csv")
aud = pd.read_csv("answers_audit_trail.csv")
assert list(ans.columns) == ["case_id","diplotype","phenotype","recommendation","abstained"]
assert len(ans) == 110 and ans.case_id.is_unique
assert set(ans.case_id) == {c["id"] for c in cases}
print("rows:", len(ans), "| abstained:", int(ans.abstained.sum()),
      "| lethal-class:", int(aud.lethal_class.sum()))
display(ans.head(12))

rows: 110 | abstained: 2 | lethal-class: 22


,case_id,diplotype,phenotype,recommendation,abstained
0,70dc9c9110f3,*4/*4,CYP2D6 poor metabolizer,codeine: avoid - diminished analgesia; if an opioid is warranted use a non-tramadol opioid,False
1,b07cff8c99c0,*1/*4,CYP2D6 intermediate metabolizer,"codeine: use label-recommended age- or weight-specific dosing; if no response and an opioid is warranted, consider a non-tramadol opioid",False
2,ce34b1a18a3d,*1/*1,CYP2D6 normal metabolizer,codeine: use label-recommended age- or weight-specific dosing,False
3,74597c5503ec,*1/*1xN,CYP2D6 ultrarapid metabolizer,codeine: avoid - risk of serious morphine toxicity; if an opioid is warranted use a non-tramadol opioid,False
4,3d0e97b86ad0,*1/*4,CYP2D6 intermediate metabolizer,"codeine: use label-recommended age- or weight-specific dosing; if no response and an opioid is warranted, consider a non-tramadol opioid",False
5,f9e4a13551fd,*4/*4,CYP2D6 poor metabolizer,"tamoxifen: avoid - use alternative hormonal therapy (aromatase inhibitor, plus ovarian suppression if premenopausal)",False
6,9d75134030af,*1/*4,CYP2D6 intermediate metabolizer,tamoxifen: consider an aromatase inhibitor instead (plus ovarian suppression if premenopausal); if an AI is contraindicated consider tamoxifen 40 mg/day and avoid CYP2D6 inhibitors,False
7,987d7bd14933,*1/*1,CYP2D6 normal metabolizer,tamoxifen: standard dosing 20 mg/day; avoid moderate/strong CYP2D6 inhibitors,False
8,61741d10f238,*1/*1xN,CYP2D6 ultrarapid metabolizer,tamoxifen: standard dosing 20 mg/day; avoid moderate/strong CYP2D6 inhibitors,False
9,c0cb8d7ad685,*4/*4,CYP2D6 poor metabolizer,ondansetron: initiate therapy with the recommended starting dose (insufficient evidence for a genotype-based change),False


### Phenotype coverage per gene

In [8]:
cov = (aud.groupby(["gene","phenotype"]).size().rename("n").reset_index()
         .sort_values(["gene","phenotype"]))
display(cov)

,gene,phenotype,n
0,CFTR,CFTR ivacaftor-non-responsive genotype (F508del homozygote),1
1,CFTR,CFTR ivacaftor-responsive genotype (G551D gating variant),1
2,CYP2B6,CYP2B6 intermediate metabolizer,1
3,CYP2B6,CYP2B6 normal metabolizer,1
4,CYP2B6,CYP2B6 poor metabolizer,1
5,CYP2B6,CYP2B6 rapid metabolizer,1
6,CYP2C19,CYP2C19 intermediate metabolizer,4
7,CYP2C19,CYP2C19 normal metabolizer,4
8,CYP2C19,CYP2C19 poor metabolizer,4
9,CYP2C19,CYP2C19 rapid metabolizer,4


### Abstentions

In [9]:
pd.set_option("display.max_colwidth", 200)
display(aud[aud.abstained][["case_id","gene","drug","diplotype","phenotype","recommendation"]])

,case_id,gene,drug,diplotype,phenotype,recommendation
102,e55c1b0dd1a0,CYP4F2,warfarin,*3/*3,CYP4F2 decreased function (*3 homozygote),warfarin: ABSTAIN - the supplied CPIC excerpt gives no CYP4F2 genotype-specific dose recommendation (its only dose table is VKORC1 x CYP2C9); a CYP4F2-only result cannot be translated into an action
103,807ece818520,CYP4F2,warfarin,*1/*1,CYP4F2 normal function,warfarin: ABSTAIN - the supplied CPIC excerpt gives no CYP4F2 genotype-specific dose recommendation (its only dose table is VKORC1 x CYP2C9); a CYP4F2-only result cannot be translated into an action


### Lethal-class cases

Cases where standard dosing of the named drug in the called phenotype can plausibly kill
or cause irreversible major organ/tissue injury — not merely treatment failure.

In [10]:
display(aud[aud.lethal_class][["case_id","gene","drug","phenotype","recommendation","lethal_rationale"]]
        .reset_index(drop=True))

,case_id,gene,drug,phenotype,recommendation,lethal_rationale
0,74597c5503ec,CYP2D6,codeine,CYP2D6 ultrarapid metabolizer,codeine: avoid - risk of serious morphine toxicity; if an opioid is warranted use a non-tramadol opioid,Ultrarapid morphine formation; documented fatal/near-fatal respiratory depression at label doses
1,ca9e14feca53,CYP2C19,clopidogrel,CYP2C19 poor metabolizer,clopidogrel: avoid if possible - use prasugrel or ticagrelor at standard dose if not contraindicated,"Loss of antiplatelet effect; stent thrombosis, MI and cardiovascular death"
2,97c3baebed4e,CYP2C9,warfarin,CYP2C9 poor metabolizer,"warfarin: use a validated genotype-guided dosing algorithm; corpus dose table gives an expected stable dose of only 0.5-2 mg/day (CYP2C9 *3/*3, assuming VKORC1 -1639 GG) - standard initiation carries a high major-bleeding risk",Stable dose 0.5-2 mg/day; standard initiation causes major/fatal haemorrhage
3,b94a43e574db,DPYD,fluorouracil,DPYD poor metabolizer,fluorouracil: avoid - do not use 5-fluorouracil or 5-fluorouracil prodrug-based regimens (complete DPD deficiency),Complete DPD deficiency; corpus states 'severe or even fatal drug toxicity'
4,c57caa44a38a,DPYD,fluorouracil,DPYD intermediate metabolizer,"fluorouracil: reduce the starting dose by 50%, then titrate on toxicity/TDM",Partial DPD deficiency; corpus states 'severe or even fatal drug toxicity' at full dose
5,6b8d2db91001,DPYD,fluorouracil,DPYD intermediate metabolizer,"fluorouracil: reduce the starting dose by 50%, then titrate on toxicity/TDM",Partial DPD deficiency; corpus states 'severe or even fatal drug toxicity' at full dose
6,c6549d637d49,DPYD,capecitabine,DPYD poor metabolizer,capecitabine: avoid - do not use 5-fluorouracil or 5-fluorouracil prodrug-based regimens (complete DPD deficiency),Complete DPD deficiency; corpus states 'severe or even fatal drug toxicity'
7,ae4a7215ae64,DPYD,capecitabine,DPYD intermediate metabolizer,"capecitabine: reduce the starting dose by 50%, then titrate on toxicity/TDM",Partial DPD deficiency; corpus states 'severe or even fatal drug toxicity' at full dose
8,08e18e809fcb,TPMT,azathioprine,TPMT poor metabolizer,"azathioprine: avoid standard dosing - consider an alternative immunosuppressant; if a thiopurine is used, drastic dose reduction is required (tolerated dose ~10% of normal)",Profound prolonged myelosuppression at conventional dose; can be fatal
9,f3dd19d8a337,NUDT15,thiopurines,NUDT15 poor metabolizer,thiopurines (azathioprine/mercaptopurine/thioguanine): avoid standard dosing - consider an alternative immunosuppressant; tolerated dose is ~10% of normal,Severe myelosuppression at conventional dose; tolerated dose ~10% of normal
